Trae los activos en tendencia alcista que esten por dar entrada segun índicador MACD.

In [15]:
# Importación.

from datos import obtener_tickers 
import pandas_ta as ta
import pandas as pd

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
# Descarga de datos temporalidades diaria semanal.

data_7d = obtener_tickers("1wk", "5y")
data_1d = obtener_tickers("1d", "3y")

Se han cargado 273 tickers...
Tiempo de descarga en 5896.79 segundos.
Se descargó datos en temporalidad 1wk con un periodo de 5y
Se han cargado 273 tickers...
Tiempo de descarga en 5925.17 segundos.
Se descargó datos en temporalidad 1d con un periodo de 3y


In [39]:
# Análisis diario.

# ==========================================
# INTERRUPTORES DE CONFIGURACIÓN (DIARIO)
# ==========================================
USAR_FILTRO_TENDENCIA_EMA = True   # Exige precio > EMA 150 y EMA 200
USAR_FILTRO_BAJO_CERO = True       # Exige MACD actual en territorio negativo
USAR_FILTRO_CONVERGENCIA = False    # Exige compresión y giro alcista inminente del histograma

VENTANA_MINIMA_DATOS = 260
PORCENTAJE_TOLERANCIA_CONVERGENCIA = 0.15

empresas_diario = []
tickers = data_1d.columns.get_level_values(0).unique()
print(f"Iniciando análisis técnico diario para {len(tickers)} activos...")

for ticker in tickers:
    try:
        df = data_1d.xs(ticker, axis=1, level=0).dropna()
        if len(df) < VENTANA_MINIMA_DATOS:
            continue

        df.ta.ema(length=150, close='Close', append=True)
        df.ta.ema(length=200, close='Close', append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        precio = df['Close'].iloc[-1]
        ema_150 = df['EMA_150'].iloc[-1]
        ema_200 = df['EMA_200'].iloc[-1]
        
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        
        hist_act = macd_act - sig_act
        hist_ant = df['MACD_12_26_9'].iloc[-2] - df['MACDs_12_26_9'].iloc[-2]

        # Evaluadores condicionales según interruptores
        pasa_tendencia = (precio > ema_150) and (precio > ema_200) if USAR_FILTRO_TENDENCIA_EMA else True
        pasa_bajo_cero = (macd_act < 0) if USAR_FILTRO_BAJO_CERO else True
        
        pasa_convergencia = (
            (hist_act < 0) and 
            (hist_act > hist_ant) and 
            (abs(hist_act) <= abs(macd_act) * PORCENTAJE_TOLERANCIA_CONVERGENCIA)
        ) if USAR_FILTRO_CONVERGENCIA else True

        if pasa_tendencia and pasa_bajo_cero and pasa_convergencia:
            print(f" MATCH DIARIO ENCONTRADO: {ticker}")
            empresas_diario.append({
                'Ticker': ticker,
                'MACD': round(macd_act, 3),
                'Señal': round(sig_act, 3),
            })
            
    except Exception as e:
        print(f"Error procesando {ticker}: {e}")
        continue
    
# OUTPUT ANÁLISIS 1D

if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print("\n" + "=" * 50)
    print("PAPELES DETECTADOS EN DIARIO:")
    print(df_final.to_string(index=False))
    print("=" * 50)
else:
    print("Sin señales diarias encontradas.")
    
print(f"Última barra: {df.index[-1]} | Barras: {len(df)}")

Iniciando análisis técnico diario para 273 activos...
 MATCH DIARIO ENCONTRADO: USO
 MATCH DIARIO ENCONTRADO: OKE
 MATCH DIARIO ENCONTRADO: KMI
 MATCH DIARIO ENCONTRADO: CLF
 MATCH DIARIO ENCONTRADO: XLE
 MATCH DIARIO ENCONTRADO: CIBR
 MATCH DIARIO ENCONTRADO: CRWD
 MATCH DIARIO ENCONTRADO: EFA
 MATCH DIARIO ENCONTRADO: TTE
 MATCH DIARIO ENCONTRADO: NEM
 MATCH DIARIO ENCONTRADO: OXY
 MATCH DIARIO ENCONTRADO: SPY
 MATCH DIARIO ENCONTRADO: DVN
 MATCH DIARIO ENCONTRADO: XBI
 MATCH DIARIO ENCONTRADO: BP
 MATCH DIARIO ENCONTRADO: XOM
 MATCH DIARIO ENCONTRADO: VLO
 MATCH DIARIO ENCONTRADO: SU
 MATCH DIARIO ENCONTRADO: NET
 MATCH DIARIO ENCONTRADO: ADI
 MATCH DIARIO ENCONTRADO: EQNR
 MATCH DIARIO ENCONTRADO: MPC
 MATCH DIARIO ENCONTRADO: VTI
 MATCH DIARIO ENCONTRADO: SLB
 MATCH DIARIO ENCONTRADO: AMD
 MATCH DIARIO ENCONTRADO: SNOW
 MATCH DIARIO ENCONTRADO: HPQ
 MATCH DIARIO ENCONTRADO: CRM
 MATCH DIARIO ENCONTRADO: STLD
 MATCH DIARIO ENCONTRADO: CVX
 MATCH DIARIO ENCONTRADO: SHOP
 MATCH DIARI

In [ ]:
# ANÁLISIS SEMANAL.
# ==========================================
# INTERRUPTORES DE CONFIGURACIÓN 1WK
# ==========================================
USAR_FILTRO_TENDENCIA_EMA = True   # Exige precio > EMA 150 y EMA 200
USAR_FILTRO_BAJO_CERO = True       # Exige MACD actual en territorio negativo
USAR_FILTRO_CONVERGENCIA = False    # Exige compresión y giro alcista inminente del histograma

VENTANA_MINIMA_DATOS = 200
PORCENTAJE_TOLERANCIA_CONVERGENCIA = 0.15

empresas_semanal = []
tickers = data_7d.columns.get_level_values(0).unique()
print(f"Iniciando análisis técnico semanal para {len(tickers)} activos...")

for ticker in tickers:
    try:
        df = data_7d.xs(ticker, axis=1, level=0).dropna()
        if len(df) < VENTANA_MINIMA_DATOS:
            continue

        df.ta.ema(length=150, close='Close', append=True)
        df.ta.ema(length=200, close='Close', append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        precio = df['Close'].iloc[-1]
        ema_150 = df['EMA_150'].iloc[-1]
        ema_200 = df['EMA_200'].iloc[-1]
        
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        
        hist_act = macd_act - sig_act
        hist_ant = df['MACD_12_26_9'].iloc[-2] - df['MACDs_12_26_9'].iloc[-2]

        # Evaluadores condicionales según interruptores
        pasa_tendencia = (precio > ema_150) and (precio > ema_200) if USAR_FILTRO_TENDENCIA_EMA else True
        pasa_bajo_cero = (macd_act < 0) if USAR_FILTRO_BAJO_CERO else True
        
        pasa_convergencia = (
            (hist_act < 0) and 
            (hist_act > hist_ant) and 
            (abs(hist_act) <= abs(macd_act) * PORCENTAJE_TOLERANCIA_CONVERGENCIA)
        ) if USAR_FILTRO_CONVERGENCIA else True

        if pasa_tendencia and pasa_bajo_cero and pasa_convergencia:
            print(f" MATCH SEMANAL ENCONTRADO: {ticker}")
            empresas_semanal.append({
                'Ticker': ticker,
                'MACD': round(macd_act, 3),
                'Señal': round(sig_act, 3),
            })
            
    except Exception as e:
        print(f"Error procesando {ticker}: {e}")
        continue
    
# OUTPUT ANÁLISIS 7D.

if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print("\n" + "=" * 50)
    print("PAPELES DETECTADOS EN SEMANAL:")
    print(df_final.to_string(index=False))
    print("=" * 50)
else:
    print("Sin señales semanal encontradas.")
    
print(f"Última barra: {df.index[-1]} | Barras: {len(df)}")


Iniciando análisis técnico semanal para 273 activos...
 MATCH SEMANAL ENCONTRADO: XLU
 MATCH SEMANAL ENCONTRADO: XME
 MATCH SEMANAL ENCONTRADO: UBER
 MATCH SEMANAL ENCONTRADO: CAT
 MATCH SEMANAL ENCONTRADO: BAC
 MATCH SEMANAL ENCONTRADO: F
 MATCH SEMANAL ENCONTRADO: EXC
 MATCH SEMANAL ENCONTRADO: CDNS
 MATCH SEMANAL ENCONTRADO: ENB
 MATCH SEMANAL ENCONTRADO: TSLA
 MATCH SEMANAL ENCONTRADO: GE
 MATCH SEMANAL ENCONTRADO: AZN
 MATCH SEMANAL ENCONTRADO: TQQQ
 MATCH SEMANAL ENCONTRADO: IWM
 MATCH SEMANAL ENCONTRADO: XLC
 MATCH SEMANAL ENCONTRADO: NEE
 MATCH SEMANAL ENCONTRADO: AMAT
 MATCH SEMANAL ENCONTRADO: AXP
 MATCH SEMANAL ENCONTRADO: MS
 MATCH SEMANAL ENCONTRADO: DBA
 MATCH SEMANAL ENCONTRADO: NOC
 MATCH SEMANAL ENCONTRADO: T
 MATCH SEMANAL ENCONTRADO: HON
 MATCH SEMANAL ENCONTRADO: INTC
 MATCH SEMANAL ENCONTRADO: BG
 MATCH SEMANAL ENCONTRADO: NXPI
 MATCH SEMANAL ENCONTRADO: WDC
 MATCH SEMANAL ENCONTRADO: C
 MATCH SEMANAL ENCONTRADO: AVGO
 MATCH SEMANAL ENCONTRADO: IBM
 MATCH SEMANAL E